# AI Model Validation: From Mathematics to Trust
## Episode 07 — Classification Metrics: Why Accuracy Can Be Completely Misleading

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

A classifier can be 95% accurate and still fail at the one task that matters.

> **There is no meaningful metric without a meaningful question.**

This episode derives binary classification evaluation from first principles: confusion matrices, prevalence, accuracy, precision, recall, specificity, FPR, FNR, F1, balanced accuracy, thresholds, ROC, precision–recall behaviour, cost-sensitive decisions, uncertainty and subgroup evidence.


## 1. Classification as a Decision

Actual class:

### Y ∈ {0, 1}

Predicted class:

### Ŷ ∈ {0, 1}

A score-producing classifier may first output:

### s(x) ∈ [0, 1]

A threshold τ creates the decision:

### Ŷ = 1 if s(x) ≥ τ, otherwise 0

The threshold is part of the deployed decision system.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys
print("Python :",sys.version.split()[0])
print("System :",platform.system(),platform.release())
print("Machine:",platform.machine())
print("NumPy  :",np.__version__)


## 2. The Confusion Matrix

Every binary decision has four possibilities:

**True Positive (TP):** actual positive, predicted positive  
**False Negative (FN):** actual positive, predicted negative  
**False Positive (FP):** actual negative, predicted positive  
**True Negative (TN):** actual negative, predicted negative

Nearly every common binary-classification metric is a different mathematical summary of these four quantities.


## 3. The 95% Accuracy Paradox

Consider 1,000 cases:

- 950 negative
- 50 positive

A classifier predicts **negative for every case**.

### TP = 0
### FN = 50
### FP = 0
### TN = 950

Accuracy becomes:

### Accuracy = (TP + TN) / (TP + TN + FP + FN)

### Accuracy = 950 / 1000 = 95%

Yet the classifier detects zero positives.


In [ ]:
TP,FN,FP,TN=0,50,0,950
N=TP+FN+FP+TN
print("Accuracy:",(TP+TN)/N)
print("Detected positives:",TP,"of",TP+FN)


> **High accuracy can coexist with complete failure on the minority class.**


## 4. Prevalence

Positive-class prevalence is:

### Prevalence = (TP + FN) / N

For our example:

### Prevalence = 50 / 1000 = 5%

A majority-only classifier can achieve:

### Accuracymajority = max(π₀, π₁)

Here that is 95%. Class imbalance can therefore make raw accuracy deceptively impressive.


## 5. Recall — Sensitivity — True Positive Rate

Recall measures the proportion of actual positives detected:

### Recall = TP / (TP + FN)

For the always-negative classifier:

### Recall = 0 / 50 = 0%

So:

### Accuracy = 95%
### Recall = 0%

There is no contradiction. The metrics answer different questions.


## 6. Specificity — True Negative Rate

Specificity measures correctly identified negatives:

### Specificity = TN / (TN + FP)

For the same classifier:

### Specificity = 950 / 950 = 100%


## 7. False Positive and False Negative Rates

### FPR = FP / (FP + TN)

### FPR = 1 − Specificity

and:

### FNR = FN / (FN + TP)

### FNR = 1 − Recall

The always-negative classifier has FNR = 100%.


## 8. Precision — Positive Predictive Value

Precision asks: of all positive predictions, how many were actually positive?

### Precision = TP / (TP + FP)

If TP + FP = 0, the ordinary formula has a zero denominator.

Precision is then mathematically undefined unless a reporting convention is explicitly adopted.


In [ ]:
def metrics(tp,fn,fp,tn):
    n=tp+fn+fp+tn
    accuracy=(tp+tn)/n if n else np.nan
    precision=tp/(tp+fp) if tp+fp else np.nan
    recall=tp/(tp+fn) if tp+fn else np.nan
    specificity=tn/(tn+fp) if tn+fp else np.nan
    fpr=fp/(fp+tn) if fp+tn else np.nan
    fnr=fn/(fn+tp) if fn+tp else np.nan
    return accuracy,precision,recall,specificity,fpr,fnr

names=["Accuracy","Precision","Recall","Specificity","FPR","FNR"]
for name,value in zip(names,metrics(0,50,0,950)):
    print(f"{name:<12}: {value}")


## 9. F1 Score

F1 is the harmonic mean of precision P and recall R:

### F1 = 2PR / (P + R)

Equivalently:

### F1 = 2TP / (2TP + FP + FN)

when the denominator is nonzero.

F1 does not explicitly include TN.


In [ ]:
pairs=[(.95,.95),(.95,.50),(.95,.10),(.60,.60)]
for p,r in pairs:
    f1=2*p*r/(p+r)
    arithmetic=(p+r)/2
    print(f"P={p:.2f} R={r:.2f} arithmetic={arithmetic:.3f} F1={f1:.3f}")


The harmonic mean is strongly influenced by the smaller value. High precision cannot completely compensate for very poor recall, or vice versa.


## 10. Balanced Accuracy

### Balanced Accuracy = (Sensitivity + Specificity) / 2

For the always-negative classifier:

### Balanced Accuracy = (0 + 1) / 2 = 50%

Compare:

### Raw Accuracy = 95%
### Balanced Accuracy = 50%

Balanced accuracy exposes what class imbalance hid.


## 11. Same Accuracy, Different Behaviour

Balanced dataset: 500 positives and 500 negatives.

**Model A:** TP=450, FN=50, FP=50, TN=450  
**Model B:** TP=400, FN=100, FP=0, TN=500

Both have 90% accuracy.


In [ ]:
models={"Model A":(450,50,50,450),"Model B":(400,100,0,500)}
for name,(tp,fn,fp,tn) in models.items():
    vals=metrics(tp,fn,fp,tn)
    print("\n"+name)
    for metric,value in zip(names,vals):
        print(f"{metric:<12}: {value:.4f}")


### AccuracyA = AccuracyB does not imply BehaviourA = BehaviourB

Model B eliminates false positives but doubles the number of missed positives.


## 12. Metrics as Conditional Probabilities

### Recall = P(Ŷ = 1 | Y = 1)

### Specificity = P(Ŷ = 0 | Y = 0)

### Precision = P(Y = 1 | Ŷ = 1)

They condition on different events. This is why precision and recall cannot be treated as interchangeable.


## 13. Base Rates and Bayes' Rule

Precision depends on prevalence:

### Precision = (Sensitivity × Prevalence) /
### [(Sensitivity × Prevalence) + FPR × (1 − Prevalence)]

Even a classifier with strong sensitivity and specificity can have modest precision when the event is rare.


In [ ]:
sensitivity=.95
specificity=.95
fpr=1-specificity

prevalences=np.logspace(-4,-.05,200)
precision=(sensitivity*prevalences)/(sensitivity*prevalences+fpr*(1-prevalences))

plt.figure(figsize=(8,5))
plt.semilogx(prevalences,precision)
plt.xlabel("Positive prevalence"); plt.ylabel("Precision")
plt.title("Precision Depends on the Base Rate")
plt.grid(alpha=.25); plt.show()

for prev in [.50,.10,.01,.001]:
    ppv=(sensitivity*prev)/(sensitivity*prev+fpr*(1-prev))
    print(f"Prevalence={prev:.3%} -> Precision={ppv:.3%}")


This matters in fraud detection, cybersecurity alerts, anomaly detection, screening and other rare-event systems.


## 14. Thresholds Change the Confusion Matrix

For a score p̂:

### Ŷτ = 1[p̂ ≥ τ]

Therefore:

### Confusion Matrix = CM(τ)

and classification metrics become threshold-dependent.


In [ ]:
rng=np.random.default_rng(2026)
n_pos,n_neg=400,1600
pos=np.clip(rng.normal(.70,.18,n_pos),0,1)
neg=np.clip(rng.normal(.30,.18,n_neg),0,1)
scores=np.concatenate([pos,neg])
actual=np.concatenate([np.ones(n_pos,dtype=int),np.zeros(n_neg,dtype=int)])

def confusion(y,s,t):
    pred=(s>=t).astype(int)
    tp=np.sum((y==1)&(pred==1))
    fn=np.sum((y==1)&(pred==0))
    fp=np.sum((y==0)&(pred==1))
    tn=np.sum((y==0)&(pred==0))
    return tp,fn,fp,tn

for t in [.2,.4,.5,.6,.8]:
    tp,fn,fp,tn=confusion(actual,scores,t)
    a,p,r,sp,_,_=metrics(tp,fn,fp,tn)
    print(f"t={t:.1f} TP={tp:3} FN={fn:3} FP={fp:3} TN={tn:4} precision={p:.3f} recall={r:.3f}")


A threshold is not merely a technical default. It determines how errors are exchanged.


## 15. Precision–Recall Trade-Off


In [ ]:
thresholds=np.linspace(0,1,201)
P=[]; R=[]; ACC=[]; F1=[]
for t in thresholds:
    tp,fn,fp,tn=confusion(actual,scores,t)
    a,p,r,sp,fpr,fnr=metrics(tp,fn,fp,tn)
    P.append(p); R.append(r); ACC.append(a)
    F1.append(2*p*r/(p+r) if np.isfinite(p) and p+r>0 else np.nan)

plt.figure(figsize=(8,5))
plt.plot(thresholds,P,label="Precision")
plt.plot(thresholds,R,label="Recall")
plt.xlabel("Threshold"); plt.ylabel("Metric")
plt.title("Threshold Changes Precision and Recall")
plt.legend(); plt.grid(alpha=.25); plt.show()


Increasing the threshold often increases precision and reduces recall, although exact behaviour depends on the score distribution.


## 16. ROC Space

At each threshold:

### TPR = Recall = TP / (TP + FN)

### FPR = FP / (FP + TN)

The ROC curve plots TPR against FPR across thresholds.


In [ ]:
TPR=[]; FPR=[]
for t in thresholds:
    tp,fn,fp,tn=confusion(actual,scores,t)
    TPR.append(tp/(tp+fn))
    FPR.append(fp/(fp+tn))

plt.figure(figsize=(7,6))
plt.plot(FPR,TPR,label="Classifier ROC")
plt.plot([0,1],[0,1],linestyle="--",label="Random-ranking reference")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(); plt.grid(alpha=.25); plt.show()


## 17. ROC AUC

A useful interpretation of AUC, with appropriate treatment of ties, is:

### AUC ≈ P(scorepositive > scorenegative)

AUC summarizes ranking behaviour across thresholds. It does not select a deployment threshold or encode application-specific error costs.


In [ ]:
comparisons=(pos[:,None]>neg[None,:]).mean()
ties=(pos[:,None]==neg[None,:]).mean()
auc=comparisons+.5*ties
print("Empirical ROC AUC:",auc)


## 18. Precision–Recall Curve

For rare positive events, precision–recall behaviour is often especially informative because it focuses directly on positive detection and positive predictions.

The precision baseline of an uninformative ranking is related to prevalence.


In [ ]:
valid=np.isfinite(P)
plt.figure(figsize=(7,6))
plt.plot(np.array(R)[valid],np.array(P)[valid])
plt.axhline(actual.mean(),linestyle="--",label="Prevalence reference")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.legend(); plt.grid(alpha=.25); plt.show()


## 19. Different Metrics Prefer Different Thresholds


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(thresholds,ACC,label="Accuracy")
plt.plot(thresholds,F1,label="F1")
plt.xlabel("Threshold"); plt.ylabel("Metric")
plt.title("Metric Choice Changes the Preferred Threshold")
plt.legend(); plt.grid(alpha=.25); plt.show()

print("Observed accuracy-maximizing threshold:",thresholds[np.nanargmax(ACC)])
print("Observed F1-maximizing threshold      :",thresholds[np.nanargmax(F1)])


### Optimal Threshold is conditional on the Validation Objective

There is no context-free best threshold.


## 20. Cost-Sensitive Classification

Let cFP be the cost of a false positive and cFN the cost of a false negative.

### C = cFP × FP + cFN × FN

Different consequences can imply different preferred thresholds.


In [ ]:
scenarios=[("Equal cost",1,1),("FN costs 10x",1,10),("FP costs 10x",10,1)]
for label,c_fp,c_fn in scenarios:
    costs=[]
    for t in thresholds:
        tp,fn,fp,tn=confusion(actual,scores,t)
        costs.append(c_fp*fp+c_fn*fn)
    best=thresholds[np.argmin(costs)]
    print(f"{label:<14}: threshold={best:.3f}, observed cost={min(costs)}")


Costs must come from application context. Mathematics can optimize a defined objective; it cannot decide what society or an organization should value.


## 21. Accuracy Has Sampling Uncertainty

If correctness Xi is Bernoulli with probability p:

### p̂ = (1/n) Σᵢ₌₁ⁿ Xᵢ

### E[p̂] = p

### Var(p̂) = p(1 − p) / n

### SE(p̂) = √[p(1 − p) / n]

The reported accuracy is itself an estimate.


## 22. Same Accuracy, Different Evidence

### 9 / 10 = 90%

### 900,000 / 1,000,000 = 90%

The point estimate is identical. The uncertainty is not.


In [ ]:
for n,p in [(10,.9),(1_000_000,.9)]:
    se=np.sqrt(p*(1-p)/n)
    print(f"n={n:>9,} accuracy={p:.1%} estimated SE={se:.7f}")


## 23. Wilson Score Interval

For observed proportion p̂, n observations and normal critical value z:

### Center = [p̂ + z²/(2n)] / [1 + z²/n]

### Half-width = [z/(1 + z²/n)] √[p̂(1−p̂)/n + z²/(4n²)]

### Interval = [Center − Half-width, Center + Half-width]

Wilson intervals generally behave better than the simple Wald interval for small n or extreme proportions.


In [ ]:
def wilson(k,n,z=1.96):
    p=k/n
    den=1+z*z/n
    center=(p+z*z/(2*n))/den
    half=(z/den)*np.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return center-half,center+half

for k,n in [(9,10),(90,100),(900,1000),(900000,1000000)]:
    lo,hi=wilson(k,n)
    print(f"{k}/{n}: estimate={k/n:.3%}, Wilson≈[{lo:.3%},{hi:.3%}]")


## 24. Uncertainty Applies to Every Metric

Precision, recall, specificity, F1, balanced accuracy and AUC are also finite-sample estimates.

Appropriate uncertainty analysis depends on the metric, evaluation design, dependence structure and sampling assumptions.

### Metric Value ≠ Exact Population Truth


## 25. Subgroup Performance

Aggregate performance can hide subgroup failure.

### Aggregate Performance ≠ Uniform Performance


In [ ]:
groups={"Group A":(470,30,40,460),"Group B":(60,40,20,80)}
for group,(tp,fn,fp,tn) in groups.items():
    a,p,r,s,_,_=metrics(tp,fn,fp,tn)
    print(group,"accuracy=",round(a,3),"recall=",round(r,3),"specificity=",round(s,3))


Subgroup analysis should be motivated by intended use, population structure, risk and governance requirements—not arbitrary slicing until an interesting result appears.


## 26. Distribution Shift Changes Metric Meaning

Precision is particularly sensitive to prevalence.

If deployment prevalence differs from evaluation prevalence, predictive values can change even when sensitivity and specificity remain similar.

### Evaluation Distribution ≠ Deployment Distribution

can imply:

### Evaluation Metric ≠ Deployment Metric


## 27. Metrics Do Not Replace Failure Analysis

Excellent aggregate metrics can coexist with catastrophic failures in narrow regions.

Validation may therefore require:

- numerical metrics
- error analysis
- subgroup analysis
- edge-case testing
- stress testing
- distribution-shift analysis
- application-specific risk assessment


## 28. Selecting the Metric

If false negatives are critical, recall may be central.

If false positives are expensive, precision or specificity may matter.

With class imbalance, balanced accuracy or class-specific metrics may be more informative than raw accuracy.

If ranking across thresholds matters, ROC or precision–recall analysis may help.

No metric becomes scientifically correct merely because it is popular.


## 29. A Defensible Classification Claim

A performance claim should specify:

- positive-class definition
- metric definition
- sample size
- prevalence
- sampling process
- decision threshold
- confusion matrix
- uncertainty
- subgroup behaviour where relevant
- consequences of FP and FN
- deployment conditions

This transforms a number into interpretable evidence.


## 30. OpenVals Perspective

A serious validation platform should not store only:

### Accuracy = 95%

It should preserve the evidence behind that number: TP, FN, FP, TN, prevalence, threshold, precision, recall, specificity, FPR, FNR, F1, balanced accuracy, ROC/PR behaviour, uncertainty, subgroup metrics, cost-sensitive analysis, evaluation distribution and metric drift.

The objective is not to maximize the number of metrics.

The objective is to determine which evidence is relevant to the trust decision.


## 31. Scientific Validation Chain

### Scores → Threshold → Confusion Matrix → Metrics → Uncertainty → Consequences → Evidence → Trust Decision


## 32. Mathematical Summary

### Accuracy = (TP + TN) / N

### Precision = TP / (TP + FP)

### Recall = TP / (TP + FN)

### Specificity = TN / (TN + FP)

### FPR = FP / (FP + TN) = 1 − Specificity

### FNR = FN / (FN + TP) = 1 − Recall

### F1 = 2PR / (P + R)

### Balanced Accuracy = (Sensitivity + Specificity) / 2

### Prevalence = (TP + FN) / N

### C = cFP × FP + cFN × FN

### Var(p̂) = p(1 − p) / n


## 33. Central Conclusion

### High Accuracy ≠ Reliable Classification

> **There is no meaningful metric without a meaningful question.**

A 95% accuracy score can describe a useful classifier. It can also describe a classifier that detects none of the events we care about.

The scientific task is not to find the metric that makes the model look best.

It is to choose measurements that expose whether the model is fit for its intended decision.

### Next Episode

**Episode 08 — Thresholds, ROC, Precision–Recall, and the Mathematics of Decision Trade-Offs**

Episode 08 will move deeper into threshold selection, ROC geometry, AUC, precision–recall curves, likelihood ratios, expected cost, decision theory and why a universal “best threshold” does not exist.
